# 🚀 Production-Ready Agents: LangChain vs LangGraph

This notebook demonstrates the key differences between LangChain and LangGraph for production deployments.

## Key Production Concerns:
- ✅ Error handling and graceful degradation
- ✅ Observability and logging
- ✅ Cost tracking and optimization
- ✅ Explicit control flow
- ✅ Testability and debugging

## Scenario:
Build a research agent that:
1. First searches local documents (RAG - fast, cheap, private)
2. Falls back to web search if needed (slower, costs API calls)
3. Handles failures gracefully

## Setup and Imports

In [ ]:
import time
from typing import TypedDict, Optional
from datetime import datetime

# LangChain imports
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_classic.chains import RetrievalQA
from langchain_core.tools import Tool
from langchain_classic.agents import initialize_agent, AgentType

# LangGraph imports
from langgraph.graph import StateGraph, END

# Web search
import wikipedia

print("✅ Imports successful")

## Metrics Tracker (Production Monitoring)

In [ ]:
class MetricsTracker:
    """Track costs and performance metrics for production monitoring"""
    
    def __init__(self):
        self.tool_calls = 0
        self.errors = 0
        self.start_time = None
        self.decisions = []
    
    def reset(self):
        self.__init__()
    
    def start(self):
        self.start_time = time.time()
    
    def log_tool_call(self, tool_name: str, success: bool):
        self.tool_calls += 1
        self.decisions.append({
            "tool": tool_name,
            "success": success,
            "timestamp": datetime.now().isoformat()
        })
        if not success:
            self.errors += 1
    
    def report(self):
        elapsed = time.time() - self.start_time if self.start_time else 0
        return {
            "tool_calls": self.tool_calls,
            "errors": self.errors,
            "elapsed_seconds": round(elapsed, 2),
            "decisions": self.decisions
        }

# Global metrics tracker
metrics = MetricsTracker()
print("✅ Metrics tracker initialized")

## Setup Local RAG System

In [ ]:
print("📚 Setting up local RAG system...")

# Load and chunk the document
loader = TextLoader("climate.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)

print(f"   Loaded {len(chunks)} chunks")

# Create embeddings and vector store
embeddings = OllamaEmbeddings(model="mxbai-embed-large")
vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    persist_directory="demo_chroma"
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3}
)

# Create QA chain
llm = Ollama(model="llama3")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

print("✅ RAG system ready")

## Define Tools with Error Handling

In [ ]:
def local_search_tool(query: str) -> str:
    """Search local documents with error handling"""
    try:
        metrics.log_tool_call("local_search", True)
        result = qa_chain.invoke({"query": query})
        return result["result"]
    except Exception as e:
        metrics.log_tool_call("local_search", False)
        return f"❌ Local search failed: {str(e)}"


def web_search_tool(query: str, simulate_failure: bool = False) -> str:
    """Search Wikipedia with error handling"""
    
    if simulate_failure:
        metrics.log_tool_call("web_search", False)
        return "❌ Web API unavailable (simulated failure)"
    
    try:
        metrics.log_tool_call("web_search", True)
        search_results = wikipedia.search(query)
        if search_results:
            page_title = search_results[0]
            summary = wikipedia.summary(page_title, sentences=3)
            return f"📰 From Wikipedia ({page_title}): {summary}"
        else:
            return "❌ No Wikipedia results found"
    except Exception as e:
        metrics.log_tool_call("web_search", False)
        return f"❌ Web search failed: {str(e)}"

print("✅ Tools defined")

---
# Part 1: LangChain Agent

**Pros:** Quick to set up, less boilerplate

**Cons:** Limited control over routing, hard to debug, black-box behavior

In [ ]:
print("="*60)
print("LANGCHAIN AGENT: Quick but Less Control")
print("="*60)

# Create tools
tools = [
    Tool(
        name="LocalSearch",
        func=local_search_tool,
        description="Search local climate documents. Use this first for climate-related questions."
    ),
    Tool(
        name="WebSearch",
        func=lambda q: web_search_tool(q, simulate_failure=False),
        description="Search Wikipedia. Use only if local search doesn't find the answer."
    )
]

# Initialize agent
llm = Ollama(model="llama3")
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,  # See the agent's thinking
    max_iterations=3,
    handle_parsing_errors=True
)

# Run query
question = "What causes the most CO2 emissions?"
print(f"\n❓ Question: {question}\n")

metrics.reset()
metrics.start()

result = agent.invoke({"input": question})
print(f"\n💬 Answer: {result['output']}\n")

# Show metrics
report = metrics.report()
print("\n📊 Metrics:")
print(f"   - Tool calls: {report['tool_calls']}")
print(f"   - Errors: {report['errors']}")
print(f"   - Time: {report['elapsed_seconds']}s")

### ⚠️ Issues with LangChain:
- Hard to predict which tool will be called
- Error handling is opaque
- Can't easily add custom retry logic
- Difficult to debug agent decisions

---
# Part 2: LangGraph Agent

**Pros:** Explicit control, observable, testable, production-ready

**Cons:** More verbose setup (but worth it for production!)

## Define Agent State

In [ ]:
class AgentState(TypedDict):
    """State for LangGraph agent"""
    question: str
    local_result: Optional[str]
    web_result: Optional[str]
    final_answer: Optional[str]
    error: Optional[str]
    next_step: Optional[str]

print("✅ State defined")

## Define Graph Nodes (Each with Clear Responsibility)

In [ ]:
def route_question(state: AgentState) -> AgentState:
    """Router: Always start with local search (cheaper, faster)"""
    print("🔀 Router: Starting with local search")
    return {**state, "next_step": "local"}


def local_search_node(state: AgentState) -> AgentState:
    """Local search with error handling"""
    print("📚 Searching local documents...")
    
    result = local_search_tool(state["question"])
    
    if "❌" not in result:
        print(f"✅ Found answer locally")
        return {
            **state,
            "local_result": result,
            "next_step": "synthesize"
        }
    else:
        print(f"⚠️  Local search failed, trying web")
        return {
            **state,
            "local_result": None,
            "next_step": "web"
        }


def web_search_node(state: AgentState) -> AgentState:
    """Web search fallback"""
    print("🌐 Searching web (fallback)...")
    
    # Change simulate_failure to True to see error handling
    result = web_search_tool(state["question"], simulate_failure=False)
    
    if "❌" not in result:
        print(f"✅ Found answer on web")
        return {
            **state,
            "web_result": result,
            "next_step": "synthesize"
        }
    else:
        print(f"❌ Web search also failed")
        return {
            **state,
            "web_result": None,
            "error": result,
            "next_step": "error"
        }


def synthesize_answer(state: AgentState) -> AgentState:
    """Combine results from all sources"""
    print("🔄 Synthesizing final answer...")
    
    sources = []
    if state.get("local_result"):
        sources.append(f"Local: {state['local_result']}")
    if state.get("web_result"):
        sources.append(f"Web: {state['web_result']}")
    
    if sources:
        return {
            **state,
            "final_answer": "\n\n".join(sources),
            "next_step": "end"
        }
    else:
        return {
            **state,
            "final_answer": "Unable to find answer",
            "next_step": "end"
        }


def handle_error(state: AgentState) -> AgentState:
    """Graceful error handling"""
    error_msg = state.get("error", "Unknown error")
    print(f"⚠️  Error: {error_msg}")
    return {
        **state,
        "final_answer": f"Sorry, couldn't retrieve an answer. Error: {error_msg}",
        "next_step": "end"
    }

print("✅ Nodes defined")

## Build and Compile the Graph

In [ ]:
# Build the graph
graph = StateGraph(AgentState)

# Add nodes
graph.add_node("router", route_question)
graph.add_node("local", local_search_node)
graph.add_node("web", web_search_node)
graph.add_node("synthesize", synthesize_answer)
graph.add_node("error_handler", handle_error)

# Set entry point
graph.set_entry_point("router")

# Add conditional edges
def route_next(state: AgentState) -> str:
    return state.get("next_step", "end")

graph.add_conditional_edges(
    "router",
    route_next,
    {"local": "local", "end": END}
)

graph.add_conditional_edges(
    "local",
    route_next,
    {"web": "web", "synthesize": "synthesize", "end": END}
)

graph.add_conditional_edges(
    "web",
    route_next,
    {"synthesize": "synthesize", "error": "error_handler", "end": END}
)

graph.add_conditional_edges(
    "synthesize",
    route_next,
    {"end": END}
)

graph.add_conditional_edges(
    "error_handler",
    route_next,
    {"end": END}
)

# Compile
app = graph.compile()

print("✅ Graph compiled and ready")

## Run the LangGraph Agent

In [ ]:
print("="*60)
print("LANGGRAPH AGENT: Production-Ready")
print("="*60)

question = "What causes the most CO2 emissions?"
print(f"\n❓ Question: {question}\n")

metrics.reset()
metrics.start()

result = app.invoke({"question": question})

print(f"\n💬 Final Answer:\n{result['final_answer']}\n")

# Show metrics
report = metrics.report()
print("\n📊 Metrics:")
print(f"   - Tool calls: {report['tool_calls']}")
print(f"   - Errors: {report['errors']}")
print(f"   - Time: {report['elapsed_seconds']}s")
print(f"\n📋 Decision Log:")
for decision in report['decisions']:
    status = "✅" if decision['success'] else "❌"
    print(f"   {status} {decision['tool']}")

---
# Key Takeaways for Production

## LangChain
✅ Fast prototyping  
✅ Less boilerplate  
❌ Hard to debug agent decisions  
❌ Limited error handling control  
❌ Unpredictable tool usage  

## LangGraph
✅ Explicit control flow (testable!)  
✅ Observable decision path  
✅ Graceful error handling  
✅ Cost-aware routing  
✅ Production monitoring ready  
❌ More verbose setup  

## Recommendation
- **Prototyping**: LangChain
- **Production**: LangGraph
- **Multi-agent systems**: CrewAI (covered in later sections)

## Production Checklist
- [ ] Error handling for all tools
- [ ] Retry logic with exponential backoff
- [ ] Structured logging for debugging
- [ ] Cost tracking per operation
- [ ] Graceful degradation when services fail
- [ ] Health checks and monitoring
- [ ] Rate limiting and circuit breakers